In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

In [3]:
# Connect to client libraries

openai = OpenAI()
OPENAI_MODEL = "gpt-5"

In [4]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '10',
  'version': '10.0.26200',
  'kernel': '10',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': ''},
 'package_managers': ['winget'],
 'cpu': {'brand': '11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz',
  'cores_logical': 16,
  'cores_physical': 8,
  'simd': []},
 'toolchain': {'compilers': {'gcc': '', 'g++': '', 'clang': '', 'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [5]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Respond with whether I need to install any C++ compiler to do this. If so, provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openai.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))
    

You do not currently have any C++ compiler installed.

Simplest way to install one (recommended: Microsoft C++ Build Tools, uses winget):
1) Open PowerShell as Administrator.
2) Run this command (non-interactive install of the C++ toolchain and Windows SDK):
   winget install --id Microsoft.VisualStudio.2022.BuildTools -e --source winget --override "--quiet --wait --norestart --add Microsoft.VisualStudio.Workload.VCTools --includeRecommended --includeOptional"
3) When it finishes, you’re ready to compile.

Optional: Verify installation by running this in a new terminal:
   "%ProgramFiles(x86)%\Microsoft Visual Studio\2022\BuildTools\VC\Auxiliary\Build\vcvars64.bat" && cl /Bv

Python compile/run commands (fastest runtime focus):
Use MSVC with high optimization, AVX2 enabled for your CPU, and link-time optimization.

- compile_command:
  ["cmd", "/d", "/s", "/c", "\"%ProgramFiles(x86)%\\Microsoft Visual Studio\\2022\\BuildTools\\VC\\Auxiliary\\Build\\vcvars64.bat\" && cl /nologo /O2 /Oi /GL /Gy /arch:AVX2 /favor:INTEL64 /DNDEBUG /EHsc /std:c++20 /MD main.cpp /link /LTCG /OPT:REF /OPT:ICF /OUT:main.exe"]

- run_command:
  ["main.exe"]

Notes:
- Place main.cpp in the current working directory of your Python script (or adjust the path in the command).
- The compile command first loads the MSVC environment (vcvars64.bat) and then compiles main.cpp into main.exe with aggressive optimizations for fastest runtime on your CPU.

In [ ]:
## include the same compile command as mentioned above

compile_command = ["cmd", "/d", "/s", "/c", "%ProgramFiles(x86)%\Microsoft Visual Studio\2022\BuildTools\VC\Auxiliary\Build\vcvars64.bat && cl /nologo /O2 /Oi /GL /Gy /arch:AVX2 /favor:INTEL64 /DNDEBUG /EHsc /std:c++20 /MD main.cpp /link /LTCG /OPT:REF /OPT:ICF /OUT:main.exe"]
run_command = ["main.exe"]                   

In [8]:
## Define the system and user prompt

system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [9]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [10]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [16]:
def port(client, model, python):
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','') ##string.replace(old,new)
    write_output(reply)

In [17]:
## python code

pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [18]:
## interpret string as python code

def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [19]:
run_python(pi)

Result: 3.141592656089
Execution Time: 25.366523 seconds


In [20]:
port(openai, OPENAI_MODEL, pi)

In [24]:
# Compile C++ file

def compile_and_run():
    subprocess.run(compile_command, check=True, text=True, capture_output=True)     ## run three times
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)

In [25]:
compile_and_run

<function __main__.compile_and_run()>